# Module 8

## Machine learning using scikit-learn

- Scikit-Learn provides efficient versions of a large number of common algorithms for:
    - Classification
    - Regression
    - Clustering
    - Dimensionality reduction
    - Model selection
    - Preprocessing
- Uniform interface: once you understand the basic for one type of model, switching to a new model or algorithm is very straightforward
- Well written documentations: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)

In [ ]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme() ## this sets some style parameters

# To check version of sklearn, run the following command in your terminal:
# python -c "import sklearn; print(sklearn.__version__)"


Dataset loading and merging: ``df_subj_FA``

In [ ]:
df_MRI = pd.read_csv('https://github.com/tractometry/AFQ-Insight/blob/main/afqinsight/data/regression_data/nodes.csv?raw=true')
df_subj = pd.read_csv('https://github.com/tractometry/AFQ-Insight/blob/main/afqinsight/data/regression_data/subjects.csv?raw=true')

df_MRI["tractID"] = df_MRI["tractID"].replace(" ", "_", regex=True)

# Transform dataset in data analysis friendly format (see Module 7)
df_FA_summary = df_MRI.groupby(["subjectID","tractID"])[["fa"]].mean().reset_index()
df_FA_summ_wide = df_FA_summary.pivot(index=["subjectID"], columns="tractID", values="fa")

df_subj_FA_wth_NA = pd.merge(df_subj, df_FA_summ_wide, on="subjectID")
df_subj_FA = df_subj_FA_wth_NA.fillna(df_subj_FA_wth_NA.mean(numeric_only=True)) # fill missing values with mean of each column
df_subj_FA = df_subj_FA.drop(columns=['Unnamed: 0', 'subjectID'])

print(f"Shape of df_subj_FA: {df_subj_FA.shape}")
print(df_subj_FA.head(n = 3))



In ``sklearn``, we typically we work with two objects:
- **Features matrix** (``X``): two-dimensional, with shape ``[n_samples, n_features]``. Usually a NumPy array or a Pandas ``DataFrame``. Features are generally real-valued, but may be Boolean or discrete-valued in some cases.
- **Target array** (``y``): *label* or *target*, with length ``n_samples``. Usually a NumPy array or Pandas ``Series``. The target array may have continuous numerical values, or discrete classes/labels.

In [ ]:
X_FA = df_subj_FA.drop(columns = ['Age', 'Gender', 'Handedness', 'IQ', 'IQ_Matrix', 'IQ_Vocab'])
X_FA.shape


In [ ]:
y = df_subj_FA['Age']
y.shape

### Basics of sklearn

1. Choose a class of models by importing the appropriate ``estimator`` class from Scikit-Learn.
2. Choose model hyperparameters by instantiating this class with desired values.
3. Arrange data into a features matrix and target vector.
4. Fit the ``estimator`` to your data by calling the ``fit()`` method.
5. Apply the model to new data:
   - For supervised learning, often we predict labels for unknown data using the ``predict()`` method.
   - For unsupervised learning, we often transform or infer properties of the data using the ``fit_transform()`` or ``predict()`` method.

### Supervised learning example I: Linear regression

As an example, let's consider a linear regression.

#### 1. Choose a class of model

In [ ]:
from sklearn.linear_model import LinearRegression

Note that other more general linear regression models exist as well; you can read more about them in the [``sklearn.linear_model`` module documentation](http://Scikit-Learn.org/stable/modules/linear_model.html).

#### 2. Choose model hyperparameters

In Scikit-Learn, hyperparameters are chosen by passing values at model instantiation.

For our linear regression example, we can instantiate the ``LinearRegression`` class and specify that we would like to fit the intercept using the ``fit_intercept`` hyperparameter:

In [ ]:
model_lin = LinearRegression(fit_intercept=True)
model_lin

In [ ]:
model_lin.get_params()

Keep in mind that when the model is instantiated, the only action is the storing of these hyperparameter values.

In particular, we have not yet applied the model to any data: ``sklearn`` makes very clear the distinction between the *choice of model* and the *application of model to data*.

#### 3. Arrange the data into a feature matrix and target vector, then split into training and test sets.


In [ ]:
from sklearn.model_selection import train_test_split

print(f"Shape of X: {X_FA.shape}")
print(f"Shape of y: {y.shape}")


# You can do it by hand or using train_test_split
# X_FA_train = X_FA[1:50]
# y_train = y[1:50]
# X_FA_test = X_FA[50:]
# y_test = y[50:]

X_FA_train, X_FA_test, y_train, y_test = train_test_split(X_FA, y, 
                                                          train_size=50,
                                                          random_state=0)
print(f"Shape of X (train): {X_FA_train.shape}")
print(f"Shape of y (train): {y_train.shape}")
print(f"Shape of X (test): {X_FA_test.shape}")
print(f"Shape of y (test): {y_test.shape}")


#### 4. Fit the model to your data

Now it is time to apply our model to the data. This can be done with the ``fit()`` method of the model.

In [ ]:
model_lin.fit(X_FA_train, y_train)

The `fit()` command performs a number of model-dependent internal computations, and the results of these computations are stored in model-specific attributes that the user can explore.

In Scikit-Learn, by convention, all model parameters learned during the `fit()` process have trailing underscores. For example, in this linear model, we have the following:

In [ ]:
model_lin.coef_

In [ ]:
model_lin.intercept_

In general, Scikit-Learn does not provide tools for inference; instead, it focuses on prediction. For inference, we can use `statsmodels`.

#### 5. Predict labels for unknown data

Once the model is trained, the main task of supervised machine learning is to evaluate it based on predictions for new data that was not part of the training set.

In Scikit-Learn, this can be done using the `predict()` method.

In [ ]:
y_train_fit = model_lin.predict(X_FA_train)
y_test_fit = model_lin.predict(X_FA_test)


print(f"Shape y (train) fit: {y_train_fit.shape}")
print(f"Shape y (test) fit:  {y_test_fit.shape}")

Finally, let's visualize the results by plotting first the raw data, and then this model fit:

In [ ]:
sns.relplot(x=y_test_fit, y = y_test)
sns.lineplot(x=np.arange(-10, 50, 1), y=np.arange(-10, 50, 1), color='g')
plt.ylabel("Actual Y")
plt.xlabel("Predicted Y");


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

print(f"Train MSE: {mean_squared_error(y_train, y_train_fit)}; Train R^2: {r2_score(y_train, y_train_fit)}")

In [ ]:
print(f"Test MSE: {mean_squared_error(y_test, y_test_fit)}; Test R^2: {r2_score(y_test, y_test_fit)}")

### Model validation

- Linear regression does not appear to perform well, so we will move away from this model.
- However, when evaluating several `estimators` (models), or different `hyperparameters` (settings of a given model), there is a risk of overfitting to the test set.
- This can happen when model choices or hyperparameters are repeatedly tweaked until the estimator performs optimally on the test set.
- As a result, the reported performance is no longer representative of how well the model will perform on new data once deployed.

To solve this problem, we can use one of the following strategies:

- Hold out another part of the dataset as a **validation set**:
  - Training is performed on the training set.
  - Model selection is performed using the validation set.
  - Final evaluation is performed once on the test set.
- Use **cross-validation**:
  - Cross-validation performs a sequence of fits in which each subset of the data is used both as a training set and as a validation set.

Visually, it looks something like this:

<div style="text-align: center;">
  <img src="https://scikit-learn.org/stable/_images/grid_search_cross_validation.png" width="400">
</div>


**Question**: Could I have guessed, using only the training data, that linear regression was going to perform poorly on unseen data?

In [ ]:
from sklearn.model_selection import cross_val_score
val_scores = cross_val_score(model_lin, X_FA_train, y_train, cv=10, scoring='r2')
val_scores.shape

In [ ]:
print(f"Train R^2: {r2_score(y_train, y_train_fit):3.2f}")
print(f"Validation R^2: {cross_val_score(model_lin, X_FA_train, y_train, cv=10).mean():3.2f}")
print(f"Test R^2: {r2_score(y_test, y_test_fit):3.2f}")

### Selecting the best model

Now that we have seen the basics of validation and cross-validation, we will look more closely at model selection and hyperparameter selection.

In general, model selection involves deciding whether to use:

- a more complicated, more flexible model
- a less complicated, less flexible model

Finding the “best” model is about balancing *bias* and *variance*.

- **Underfitting** = high bias, low variance
- **Overfitting** = low bias, high variance

Simple linear regression does not allow us to explicitly control model complexity. However, its extension, **Lasso regression**, does.

Lasso solves the following problem:

$$
\hat{\beta}
=
\arg\min_{\beta}
\left[
\frac{1}{2n} \lVert y - X\beta \rVert_2^2
+
\alpha \lVert \beta \rVert_1
\right]
$$

Here, $\alpha$ controls the strength of regularization:

- larger $\alpha$ → simpler model with more shrinkage
- smaller $\alpha$ → more flexible model with less shrinkage


In [ ]:
from sklearn.linear_model import Lasso

# Create a Lasso model
model_lasso = Lasso(alpha=0.0001)
# Fit model to data 
model_lasso.fit(X_FA_train, y_train)

y_train_fit = model_lasso.predict(X_FA_train)

print(f"Train R^2: {r2_score(y_train, y_train_fit)}")

It may be tempting to check performance on the test set and optimize the choice of `alpha` based on that performance, but this should be avoided. Instead we use crossvalidation on training data.

In [ ]:
model = Lasso(alpha = 0.1)

# Perform cross-validation (e.g., with 5 folds) using mean squared error
r2_val_scores = cross_val_score(model, X_FA_train, y_train, cv=10, scoring="r2")

print("Validation MSE:",r2_val_scores.mean())

Let's explore more systematically how the choice of `alpha` affects training error and cross-validated error.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import validation_curve

# Candidate alpha values for Lasso
alpha_grid = np.logspace(-4, 2, 50)

model = Lasso(max_iter=10000)

train_r2, val_r2 = validation_curve(
    model,
    X_FA_train,
    y_train,
    param_name="alpha",
    param_range=alpha_grid,
    cv=10,
    scoring="r2"
)

val_r2.shape

In [ ]:
plt.semilogx(alpha_grid, np.mean(train_r2, axis=1), label="training MSE")
plt.semilogx(alpha_grid, np.mean(val_r2, axis=1), label="validation MSE")

plt.xlabel("alpha")
plt.ylabel("R^2")
plt.title("Lasso validation curve")
plt.show()

In [ ]:
# Optimal hyperparameter

best_alpha = alpha_grid[val_r2.mean(axis=1).argmax()]

# Compute test MSE for the best alpha value
model_lasso_best = Lasso(alpha=best_alpha, max_iter=10000)

# For demonstration purposes only, let's check the test error
y_test_fit = model_lasso_best.fit(X_FA_train, y_train).predict(X_FA_test)
print(f"Test R^2: {r2_score(y_test, y_test_fit)}")

### Validation in practice: grid search

Using `validation_curve` was meant to give you some intuition about the trade-off between bias and variance.

In practice, models generally have more than one hyperparameter to tune, so plots of validation and learning curves change from lines to multidimensional surfaces, which are difficult to visualize.

Scikit-Learn provides automated tools for hyperparameter search (meta-estimators), including the `GridSearchCV`

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid_lasso = {'alpha': alpha_grid,
                    'fit_intercept': [True, False]}

grid = GridSearchCV(Lasso(max_iter=10000), param_grid_lasso, cv=10, refit=True, scoring='r2')

Notice that like any sklearn estimator, this has not yet been applied to any data.
Calling the ``fit()`` method will fit the model at each grid point, keeping track of the scores along the way:

In [ ]:
grid.fit(X_FA_train, y_train)

Now that the model has been fit, we can retrieve the best parameters.

In [ ]:
grid.best_params_

In [ ]:
grid.best_score_

Or we can directly retrieve the best model.

In [ ]:
# For demonstration purposes only, let's check the test error

model_lasso_best = grid.best_estimator_

y_test_fit = model_lasso_best.fit(X_FA_train, y_train).predict(X_FA_test)
plt.scatter(y_test, y_test_fit)
plt.plot(np.arange(0,30,1), np.arange(0,30,1))
plt.ylabel("Actual Y")
plt.xlabel("Predicted Y");

# Suspicious! We have a tiny test set here
print(f"Test R^2: {r2_score(y_test, y_test_fit)}")

### Defining a pipeline by combining models and/or preprocessing steps

For instance, with Lasso regression, it is often advised to standardize the features of your regression matrix so that the penalty is applied fairly across features.

This matters because Lasso penalizes the size of the coefficients. If features are measured on very different scales, then the penalty can affect some coefficients more strongly than others simply because of the units of measurement.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Candidate alpha values for Lasso
alpha = np.logspace(-4, 2, 50)

# Lasso works better when features are scaled
model_std_lasso = make_pipeline(
    StandardScaler(),
    Lasso(max_iter=10000)
)

model_std_lasso.named_steps

In [ ]:
param_grid_std_lasso = {
              'lasso__alpha': alpha_grid,
              'lasso__fit_intercept': [True, False]}

grid = GridSearchCV(model_std_lasso, param_grid_std_lasso, cv=10, scoring='r2')

grid.fit(X_FA_train, y_train)

In [ ]:
grid.best_score_

In [ ]:
# For demonstration purposes only, let's check the test error

model_std_lasso_best = grid.best_estimator_

y_test_fit = model_std_lasso_best.fit(X_FA_train, y_train).predict(X_FA_test)
plt.scatter(x = y_test, y = y_test_fit)
plt.plot(np.arange(0,30,1), np.arange(0,30,1))
plt.ylabel("Actual Y")
plt.xlabel("Predicted Y");

print(f"Test R^2: {r2_score(y_test, y_test_fit)}")

### Feature Engineering

The previous sections outlined the main steps of a supervised analysis, assuming numerical data in a tidy `[n_samples, n_features]` format. In the real world, data may come in different forms, such as unstructured data.

One way to deal with this is through *feature engineering*: creating numerical features from raw data in order to build a feature matrix.

Here, we will look at one special case: creating additional features from existing numerical features, for instance by adding polynomial and interaction terms.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
X_FA_extended = poly.fit_transform(X_FA)

print(f"From {X_FA.shape} to {X_FA_extended.shape}")

For degree 2, and for each pair of features $x_1$ and $x_2$, the `PolynomialFeatures` transform adds features representing $x_1^2$, $x_2^2$, and $x_1x_2$.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Candidate alpha values for Lasso
alpha_grid = np.logspace(-4, 2, 50)

# Lasso works better when features are scaled
model_std_poly_lasso = make_pipeline(
    PolynomialFeatures(include_bias=False),
    StandardScaler(),
    Lasso(max_iter=10000)
)

model_std_poly_lasso.named_steps

In [ ]:
param_grid_std_poly_lasso = {
              'polynomialfeatures__degree': [1,2],
              'lasso__alpha': alpha_grid,
              'lasso__fit_intercept': [True, False]}

grid = GridSearchCV(model_std_poly_lasso, param_grid_std_poly_lasso, cv=10, scoring="r2", refit=True)
grid.fit(X_FA_train, y_train)

print(grid.best_estimator_)
print(grid.best_score_)


In [ ]:
# For demonstration purposes only, let's check the test error

model_std_poly_lasso = grid.best_estimator_

y_test_fit = model_std_poly_lasso.predict(X_FA_test)
plt.scatter(x = y_test, y = y_test_fit)
plt.plot(np.arange(0,30,1), np.arange(0,30,1))

print(f"Test R^2: {r2_score(y_test, y_test_fit)}")

### Imputation of Missing Data

Another commonly needed feature in machine learning is the *imputation* of missing values. Several imputation strategies are natively supported in Scikit-Learn.

For baseline imputation approaches, such as using the mean, median, or most frequent value, Scikit-Learn provides the `SimpleImputer` class. Other related classes, such as `KNNImputer`, support more flexible imputation strategies.

In [ ]:
from sklearn.impute import KNNImputer
imp = KNNImputer()

X_FA_wth_NA = df_subj_FA_wth_NA.drop(columns = ['Unnamed: 0', 'subjectID', 'Age', 'Gender', 'Handedness', 'IQ', 'IQ_Matrix', 'IQ_Vocab'])

X_FA_imp = imp.fit_transform(X_FA_wth_NA)
X_FA_imp

The imputed data can then be fed directly into an estimator, such as `LinearRegression`, as part of a `Pipeline`.

### Final regression analysis

For simplicity, we previously summarized measurements at the tract level by reducing 100 measurements to 1. This allowed us to run a simple linear regression.

With Lasso, however, we do not have to reduce the data in this way. Let's run a more involved analysis.

In [ ]:
# Transform dataset
df_FA_HD_wide = df_MRI.pivot(index=["subjectID"], columns=["tractID", "nodeID"] , values="fa")
df_FA_HD_wide.columns = df_FA_HD_wide.columns.to_flat_index()
df_subj_FA_HD = pd.merge(df_subj, df_FA_HD_wide, on="subjectID")
df_subj_FA_HD = df_subj_FA_HD.drop(columns=['Unnamed: 0', 'subjectID'])

X_FA_HD = df_subj_FA_HD.drop(columns = ['Age', 'Gender', 'Handedness', 'IQ', 'IQ_Matrix', 'IQ_Vocab'])

X_FA_HD_train, X_FA_HD_test, y_train, y_test = train_test_split(X_FA_HD, y, 
                                                                train_size=50,
                                                                random_state=0)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

# Let's use Ridge regression instead of Lasso. It's a bit faster but we renounce to the variable selection feature of Lasso

# Candidate alpha values for Ridge
alpha_grid = np.logspace(-4, 2, 50)

# Ridge works better when features are scaled
model_HD_ridge = make_pipeline(
    SimpleImputer(strategy="median"),
    Ridge(max_iter=10000)
)

model_HD_ridge.named_steps

In [ ]:
param_grid_HD = {
              'ridge__alpha': alpha_grid,
              'ridge__fit_intercept': [True, False]}

grid = GridSearchCV(model_HD_ridge, param_grid_HD, cv=10, scoring="r2", refit=True)
grid.fit(X_FA_HD_train, y_train)

print(grid.best_estimator_)
print(grid.best_score_)

In [ ]:
# For demonstration purposes only, let's check the test error

model_HD_ridge_log_age = grid.best_estimator_

y_test_fit = model_HD_ridge_log_age.fit(X_FA_HD_train, y_train).predict(X_FA_HD_test)
plt.scatter(y_test, y_test_fit)
plt.plot(np.arange(0,30,1), np.arange(0,30,1))

# Suspicious! We have a tiny test set here
print(f"Test R^2: {r2_score(y_test, y_test_fit)}")